In [1]:
from pathlib import Path
import pandas as pd, numpy as np, json, re, os
p=Path('/workspace/bpx68c_Signal1.csv')
print('\n'.join(p.read_text(encoding='latin-1').splitlines()[:14]))
print('PROJECT_ID', os.environ.get('PROJECT_ID'))

TP4 Time Domain Export File
Generated From: C:\TP4_46\DataStore\EventDatabase.TP4_db
Recording Session ID: bpx68c - Polyurethane Rubber - Further Tests
Recording Session Date: 07/30/2026 01:04:42 PM,
Event: 1
EventTime: 7/30/2026 7:08:32 PM
Order: 0

Time (sec), CH2 Acc (G's), CH3 Acc (G's), CH4 Acc (G's), CH5 Acc (G's), 
0.000000E+000, -8.379204E-001, 2.932870E+000, 1.547772E+000, 1.077274E+000
8.000000E-007, -3.800876E-001, 2.943594E+000, 1.427588E+000, 9.827168E-001
1.600000E-006, -4.681988E-001, 2.966828E+000, 1.810877E+000, 1.654747E+000
2.400000E-006, -8.966612E-001, 2.230483E+000, 2.018762E+000, 2.007084E+000
3.200000E-006, -1.831331E+000, 3.270660E+000, 1.840111E+000, 2.329028E+000
PROJECT_ID None


In [2]:
import numpy as np, pandas as pd
from scipy import signal, integrate, stats
from pathlib import Path

def load(n):
    d=np.loadtxt(f'/workspace/bpx68c_Signal{n}.csv', delimiter=',', skiprows=9, usecols=range(5))
    return d[:,0],d[:,1:]
for n in [1,10,11,20,22,31,32,41]:
    t,x=load(n); y=x[:,3]
    im=np.argmax(np.abs(y))
    print(n, len(t), 1/np.median(np.diff(t)), y[:10].round(1), 'raw peak',y[im], 'at ms', t[im]*1e3, 'tail med',np.median(y[-6250:]))

1 25000 1249999.9999999697 [1.1 1.  1.7 2.  2.3 3.2 3.  3.  3.8 4.1] raw peak 1617.905 at ms 0.6464 tail med -0.1418354
10 25000 1249999.9999999697 [ 9.6  9.9 10.1 10.1 11.3 10.8 10.4 11.1 11.2 10.8] raw peak 2360.011 at ms 0.8512 tail med -0.9523235
11 25000 1249999.9999999697 [24.3 25.  25.5 25.2 24.6 24.7 24.9 24.3 24.  23.7] raw peak 587.1074 at ms 1.764 tail med -0.23808085
20 25000 1249999.9999999697 [ 5.1  5.9  5.9  6.7  8.9  9.9 11.3 13.4 14.5 15.5] raw peak 508.7051 at ms 1.7288 tail med 0.05628389
22 25000 1249999.9999999697 [7.1 7.3 7.1 6.7 7.2 6.6 6.3 6.5 6.1 6.4] raw peak 303.6437 at ms 0.3904 tail med -0.8453841
31 25000 1249999.9999999697 [5.3 5.1 5.6 5.7 5.3 5.1 5.5 5.7 5.5 6.1] raw peak 247.7639 at ms 0.4304 tail med -0.1530922
32 25000 1249999.9999999697 [14.6 15.  14.7 15.3 15.7 15.7 15.9 15.6 15.7 15.8] raw peak 224.2508 at ms 0.9768000000000001 tail med -0.7902259
41 25000 1249999.9999999697 [4.8 4.9 5.1 5.5 4.4 4.2 4.9 4.8 4.6 4.4] raw peak 247.2383 at ms 2.2568 t

In [3]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy import signal, integrate, stats

cfg_map={**{i:'A' for i in range(1,11)}, **{i:'B' for i in range(11,21)}, **{i:'C' for i in range(22,32)}, **{i:'D' for i in range(32,42)}}
records={}
for n,c in cfg_map.items():
    d=np.loadtxt(f'/workspace/bpx68c_Signal{n}.csv',delimiter=',',skiprows=9,usecols=range(5))
    records[n]=(d[:,0],d[:,1:],c)
fs=1/np.median(np.diff(records[1][0])); print('records',len(records),'fs',fs)

def filt(x, fc, order=2, padtype='odd', padlen=None):
    sos=signal.butter(order,fc,btype='low',fs=fs,output='sos')
    return signal.sosfiltfilt(sos,x,padtype=padtype,padlen=padlen)
# compare likely definitions to published CFC180 values for signal 1 and aggregate
pub=json.load(open('/workspace/pu_configs_metrics.json'))
pubrows={r['signal']:r for rows in pub['per_capture'].values() for r in rows}
for order in [2,4]:
 for fc in [180*1.6,180*1.65,180*1.667]:
    vals=[]
    for n,(t,x,c) in records.items():
      z=x[:,3]-np.median(x[:,3]); vals.append(np.max(np.abs(filt(z,fc,order)[:15000]))-pubrows[n]['ch5_180_g'])
    print(order,fc,'diff mean/maxabs',np.mean(vals),np.max(np.abs(vals)))

records 40 fs 1249999.9999999697
2 288.0 diff mean/maxabs -4.892294435911327 13.175995094142479
2 297.0 diff mean/maxabs -1.2026283250154974 3.2733274359534335


2 300.06 diff mean/maxabs 0.02391703508245726 0.06532558936811483
4 288.0 diff mean/maxabs -3.6687301662662066 28.374221257144995
4 297.0 diff mean/maxabs 0.08370994729182187 18.294782446817976
4 300.06 diff mean/maxabs 1.3285796140041761 14.887380375898829


In [4]:
# Independent pipeline: tail baseline; 2-pole Butterworth applied forward/backward; full-record peak search.
from scipy import signal, integrate, stats
import numpy as np, pandas as pd, math

def fwhm(t,y,i):
    s=1 if y[i]>=0 else -1; z=s*y; h=z[i]/2
    lo=i
    while lo>0 and z[lo-1]>=h: lo-=1
    hi=i
    while hi<len(z)-1 and z[hi+1]>=h: hi+=1
    # linear-crossing interpolation where available
    tl=t[lo]
    if lo>0 and z[lo]!=z[lo-1]: tl=t[lo-1]+(h-z[lo-1])*(t[lo]-t[lo-1])/(z[lo]-z[lo-1])
    th=t[hi]
    if hi<len(z)-1 and z[hi+1]!=z[hi]: th=t[hi]+(h-z[hi])*(t[hi+1]-t[hi])/(z[hi+1]-z[hi])
    return 1e3*(th-tl), lo==0

def independent_metrics(n,t,x,c,padtype='odd'):
    # Last 5 ms medians: avoids treating the impact pulse as baseline.
    base=np.median(x[t>=0.015],axis=0); xb=x-base
    top=xb[:,:3]; inp=xb[:,3]
    out={'signal':n,'config':c}
    imax=int(np.searchsorted(t,0.012))
    for tag,fc in [('raw',None),('1000',1667.0),('180',300.0)]:
        if fc is None: fi=inp; ft=top
        else:
            sos=signal.butter(2,fc,fs=fs,output='sos')
            fi=signal.sosfiltfilt(sos,inp,padtype=padtype)
            ft=np.column_stack([signal.sosfiltfilt(sos,top[:,j],padtype=padtype) for j in range(3)])
        res=np.linalg.norm(ft,axis=1)
        ii=np.argmax(np.abs(fi[:imax])); io=np.argmax(res[:imax])
        out[f'in_{tag}']=abs(fi[ii]); out[f'out_{tag}']=res[io]; out[f'T_{tag}']=res[io]/abs(fi[ii])
        if tag=='180':
            out['width_ms'],out['width_left_trunc']=fwhm(t,fi,ii)
            out['onset_frac']=abs(fi[0])/abs(fi[ii])
            v=integrate.cumulative_trapezoid(fi*9.80665,t,initial=0)
            out['dv_cap_max']=np.max(np.abs(v)); out['dv_cap_final']=abs(v[-1])
            out['t_in_peak_ms']=t[ii]*1e3; out['t_out_peak_ms']=t[io]*1e3
    return out
rows=[]
for n,(t,x,c) in records.items(): rows.append(independent_metrics(n,t,x,c))
df=pd.DataFrame(rows).sort_values('signal')
metrics=['in_raw','out_raw','T_raw','in_1000','out_1000','T_1000','in_180','out_180','T_180','width_ms','dv_cap_max','dv_cap_final','onset_frac']
summary=df.groupby('config')[metrics].agg(['mean','std'])
for m in metrics: summary[(m,'cv_pct')]=100*summary[(m,'std')]/summary[(m,'mean')]
print(df.head().round(4).to_string(index=False))
print('\nIndependent summary')
show=[]
for c,g in df.groupby('config'):
 d={'config':c}
 for m in metrics: d[m]=g[m].mean(); d[m+'_cv']=100*g[m].std(ddof=1)/g[m].mean()
 show.append(d)
sumdf=pd.DataFrame(show).set_index('config')
print(sumdf[['in_raw','in_raw_cv','out_raw','out_raw_cv','in_1000','in_1000_cv','out_1000','out_1000_cv','T_1000','T_1000_cv','in_180','in_180_cv','out_180','out_180_cv','T_180','T_180_cv','width_ms','width_ms_cv','dv_cap_max','dv_cap_max_cv','onset_frac']].round(4).to_string())
print('left truncated counts',df.groupby('config').width_left_trunc.sum().to_dict())

 signal config    in_raw   out_raw  T_raw  in_1000  out_1000  T_1000   in_180  out_180  T_180  width_ms  width_left_trunc  onset_frac  dv_cap_max  dv_cap_final  t_in_peak_ms  t_out_peak_ms
      1      A 1618.0468 1739.9112 1.0753 896.1340  919.6825  1.0263 380.1584 400.4848 1.0535    1.5985              True      0.5418      5.5812        5.2371        0.7712         0.8440
      2      A 1665.3845 1950.7788 1.1714 850.1682  937.6652  1.1029 375.6205 389.8388 1.0379    1.6319              True      0.5375      5.6155        5.2514        0.7936         0.8720
      3      A 1800.6688 2085.1311 1.1580 832.1087  942.1178  1.1322 373.3644 389.2832 1.0426    1.6439              True      0.5296      5.6127        5.2582        0.8040         0.8824
      4      A 1947.5508 2163.3828 1.1108 818.4899  962.1611  1.1755 371.5652 387.5253 1.0430    1.6519              True      0.5299      5.6107        5.2667        0.8096         0.8872
      5      A 1989.7196 2188.2475 1.0998 814.0179  963

In [5]:
# Edge-condition sensitivity: same tail-baselined data, alternate filtfilt endpoint rules.
def edge_variant(n,fc,pad):
 t,x,c=records[n]; xb=x-np.median(x[t>=.015],axis=0); sos=signal.butter(2,fc,fs=fs,output='sos')
 if pad=='none':
  fi=signal.sosfiltfilt(sos,xb[:,3],padtype=None,padlen=0)
  ft=np.column_stack([signal.sosfiltfilt(sos,xb[:,j],padtype=None,padlen=0) for j in range(3)])
 else:
  fi=signal.sosfiltfilt(sos,xb[:,3],padtype=pad)
  ft=np.column_stack([signal.sosfiltfilt(sos,xb[:,j],padtype=pad) for j in range(3)])
 m=t<.012; pi=np.max(np.abs(fi[m])); po=np.max(np.linalg.norm(ft[m],axis=1)); return pi,po,po/pi
ers=[]
for fcname,fc in [('180',300),('1000',1667)]:
 for pad in ['odd','even','constant','none']:
  for n in records:
   pi,po,tr=edge_variant(n,fc,pad); ers.append((n,cfg_map[n],fcname,pad,pi,po,tr))
ed=pd.DataFrame(ers,columns=['signal','config','band','pad','input','output','T'])
res=ed.groupby(['band','config','pad']).agg(T_mean=('T','mean'),T_cv=('T',lambda x:100*x.std(ddof=1)/x.mean()),in_mean=('input','mean'),out_mean=('output','mean')).reset_index()
print(res.round(4).to_string(index=False))
# Range of each drop across endpoint rules; config mean relative spans
sp=ed.groupby(['band','config','signal']).T.agg(lambda x:100*(x.max()-x.min())/x.mean()).groupby(['band','config']).agg(['mean','max'])
print('\nWithin-drop T span across endpoint rules (%)\n',sp.round(2))

band config      pad  T_mean   T_cv  in_mean  out_mean
1000      A constant  1.1782 6.1298 813.1188  955.4726
1000      A     even  1.1782 6.1299 813.1192  955.4726
1000      A     none  1.1782 6.1298 813.1188  955.4726
1000      A      odd  1.1782 6.1296 813.1185  955.4726
1000      B constant  1.0218 1.1934 343.4209  351.0091
1000      B     even  1.0218 1.1934 343.4209  351.0091
1000      B     none  1.0218 1.1934 343.4209  351.0091
1000      B      odd  1.0218 1.1934 343.4209  351.0091
1000      C constant  1.1316 0.9027 182.6021  206.6459
1000      C     even  1.1316 0.9027 182.6021  206.6459
1000      C     none  1.1316 0.9027 182.6021  206.6459
1000      C      odd  1.1316 0.9027 182.6021  206.6459
1000      D constant  1.1250 1.2006 194.8559  219.1664
1000      D     even  1.1250 1.2006 194.8559  219.1664
1000      D     none  1.1250 1.2006 194.8559  219.1664
1000      D      odd  1.1250 1.2006 194.8559  219.1664
 180      A constant  1.0445 0.4440 370.1689  386.6368
 180      

In [6]:
# Baseline-choice sensitivity, including exact published convention.
def metrics_baseline(basekind):
 rr=[]
 for n,(t,x,c) in records.items():
  if basekind=='full_median': b=np.median(x,axis=0)
  elif basekind=='tail5ms_median': b=np.median(x[t>=.015],axis=0)
  elif basekind=='tail2ms_mean': b=np.mean(x[t>=.018],axis=0)
  xb=x-b
  d={'signal':n,'config':c}
  for tag,fc in [('180',300.06),('1000',1667)]:
   sos=signal.butter(2,fc,fs=fs,output='sos'); fi=signal.sosfiltfilt(sos,xb[:,3])
   ft=np.column_stack([signal.sosfiltfilt(sos,xb[:,j]) for j in range(3)])
   m=t<.012; d['T'+tag]=np.max(np.linalg.norm(ft[m],axis=1))/np.max(np.abs(fi[m])); d['i'+tag]=np.max(abs(fi[m]));d['o'+tag]=np.max(np.linalg.norm(ft[m],axis=1))
  rr.append(d)
 return pd.DataFrame(rr)
bs=[]
for bk in ['full_median','tail5ms_median','tail2ms_mean']:
 q=metrics_baseline(bk)
 for c,g in q.groupby('config'):
  bs.append([bk,c,*sum(([g[m].mean(),100*g[m].std(ddof=1)/g[m].mean()] for m in ['i180','o180','T180','i1000','o1000','T1000']),[])])
cols=['baseline','config']+sum(([m+'_mean',m+'_cv'] for m in ['i180','o180','T180','i1000','o1000','T1000']),[])
bdf=pd.DataFrame(bs,columns=cols)
print(bdf.round(4).to_string(index=False))

      baseline config  i180_mean  i180_cv  o180_mean  o180_cv  T180_mean  T180_cv  i1000_mean  i1000_cv  o1000_mean  o1000_cv  T1000_mean  T1000_cv
   full_median      A   370.6745   1.4466   378.9045   1.5079     1.0222   0.4296    813.5839    4.8235    946.8758    1.7894      1.1669    6.1807
   full_median      B   261.4381   1.4949   260.4947   1.5536     0.9964   0.3449    344.1630    3.9434    341.1094    4.9102      0.9908    1.3739
   full_median      C   174.3357   1.6837   171.9132   1.0580     0.9862   0.9527    182.6032    1.8184    196.2396    2.3326      1.0746    0.9318
   full_median      D   183.5209   1.7652   181.4433   1.5242     0.9887   0.4950    194.7506    3.3316    209.2719    2.8618      1.0748    1.2033
tail5ms_median      A   370.2091   1.4199   386.7216   1.5182     1.0446   0.4470    813.1185    4.8144    955.4726    1.7402      1.1782    6.1296
tail5ms_median      B   260.6960   1.5217   270.4296   1.4219     1.0374   0.2756    343.4209    3.9762    351.0

In [7]:
import os
from e14c.edison_client import get_edison_client
from e14c.subagents import submit_task, check_task
client=get_edison_client()
project_id=await client.acreate_project(name='pu-configs-adversarial-review', description='Independent reanalysis and adversarial review of a polyurethane absorber arrangement sweep for a benchtop drop tower.')
standards_task=await submit_task('''Provide a source-grounded technical review for a drop-tower shock analysis. Focus on: (1) SAE J211 Channel Frequency Class filtering, whether phaseless forward-backward filtering is standard and risks when records begin mid-pulse; (2) accelerometer mounting and mass-loading checks under ISO 5347 and ISO 16063, especially wax/adhesive mounting on a compliant lightweight tip; (3) shock response spectrum practice in MIL-STD-810 Method 516 and IEST guidance, including half-sine duration selection; (4) relevance and limitations of ASTM D3332 and ASTM D7136 for this rig. Verify exact standard titles/editions where possible. Distinguish direct requirements from informed recommendations. Return traceable citations/URLs; do not guess clauses.''',project_id=project_id,agent_type='data-retrieval')
print(project_id, standards_task)

5223cf63-fcbf-4e47-ad6c-f9d6fbb91b64 74990183-f4d6-4b25-8d07-dd3a48f338d6


In [8]:
# Frequency-domain audit using linear channel spectra, not spectrum of the nonlinear resultant.
# Use common 0-20 ms record, remove tail median, linear detrend, Tukey taper.
from scipy.signal import windows
spec_rows=[]
fft_cache={}
for n,(t,x,c) in records.items():
 xb=x-np.median(x[t>=.015],axis=0)
 xd=signal.detrend(xb,axis=0,type='linear')
 w=windows.tukey(len(t),alpha=.2); X=np.fft.rfft(xd*w[:,None],axis=0); f=np.fft.rfftfreq(len(t),1/fs)
 # Parseval-proportional energies, common scaling cancels.
 P=np.abs(X)**2
 total=(f>=100)&(f<=20000); band=(f>=450)&(f<=800)
 outP=P[:,:3].sum(axis=1); inpP=P[:,3]
 spec_rows.append({'signal':n,'config':c,'out_band_frac':outP[band].sum()/outP[total].sum(),
                   'in_band_frac':inpP[band].sum()/inpP[total].sum(),
                   'out_band_abs':outP[band].sum(),'in_band_abs':inpP[band].sum(),
                   'band_energy_ratio':outP[band].sum()/inpP[band].sum(),
                   'out_peakf':f[band][np.argmax(outP[band])], 'in_peakf':f[band][np.argmax(inpP[band])]})
 fft_cache[n]=(f,X)
sdf=pd.DataFrame(spec_rows)
print(sdf.groupby('config').agg({k:['mean','std'] for k in sdf.columns if k not in ['signal','config']}).round(4).to_string())
# Normalize absolute band energies to A for interpretable ordering
for col in ['out_band_abs','in_band_abs','band_energy_ratio']:
 m=sdf.groupby('config')[col].mean(); print(col,'relative A', (m/m['A']).round(3).to_dict())
# Ensemble frequency-response estimator and multiple-output coherence across 10 nominal replicates.
ens=[]
for c,ns in {c:[n for n,v in cfg_map.items() if v==c] for c in 'ABCD'}.items():
 Xs=np.stack([fft_cache[n][1] for n in ns])
 U=Xs[:,:,3]; Y=Xs[:,:,:3]
 Suu=np.mean(abs(U)**2,axis=0); Syy=np.mean(abs(Y)**2,axis=0); Syu=np.mean(Y*np.conj(U[:,:,None]),axis=0)
 coh=np.abs(Syu)**2/(Syy*Suu[:,None])
 H=Syu/Suu[:,None]
 band=(f>=450)&(f<=800)
 # input-power weighted band gain and coherence, tri-axis sum
 gain=np.sqrt(np.sum(np.abs(H[band])**2*Suu[band,None])/np.sum(Suu[band]))
 cohw=np.sum(coh[band]*Syy[band],axis=0)/np.sum(Syy[band],axis=0)
 ens.append([c,gain,*cohw,coh[band].mean(),f[band][np.argmax(Syy[band].sum(axis=1))],f[band][np.argmax(Suu[band])]])
print('\nEnsemble H1-like band results')
print(pd.DataFrame(ens,columns=['config','tri_gain','coh_ch2','coh_ch3','coh_ch4','coh_unweighted','out_peak_Hz','in_peak_Hz']).round(3).to_string(index=False))

       out_band_frac         in_band_frac          out_band_abs                 in_band_abs               band_energy_ratio         out_peakf          in_peakf         
                mean     std         mean     std          mean           std          mean           std              mean     std      mean      std     mean      std
config                                                                                                                                                                  
A             0.3065  0.0341       0.1787  0.0209  1.322063e+12  1.269323e+11  1.909401e+11  3.453911e+09            6.9311  0.7349     505.0  15.8114    500.0   0.0000
B             0.0695  0.0105       0.0700  0.0097  6.710360e+10  1.302107e+10  5.105615e+10  9.154974e+09            1.3130  0.0690     500.0   0.0000    500.0   0.0000
C             0.0140  0.0018       0.0036  0.0007  4.763365e+09  6.489947e+08  7.674931e+08  1.112044e+08            6.3728  1.4697     745.0  92.6463    5

In [9]:
# Audit the premise that t=0 is physically mid-pulse, and emulate alternate trigger-aligned crops for A/B.
triggers={'A':300,'B':300,'C':150,'D':150}
edge=[]
for n,(t,x,c) in records.items():
 y=x[:,3]-np.median(x[t>=.015,3]); trig=triggers[c]
 cross=np.flatnonzero(np.abs(y)>=trig)
 edge.append([n,c,y[0],np.mean(y[:100]),np.std(y[:100]),t[cross[0]]*1e3 if len(cross) else np.nan,np.max(abs(y)),trig])
edge_df=pd.DataFrame(edge,columns=['signal','config','raw_t0','raw_first80us_mean','raw_first80us_sd','first_trigger_cross_ms','raw_peak','trigger'])
print(edge_df.groupby('config').agg({k:['mean','min','max'] for k in edge_df.columns if k not in ['signal','config']}).round(3).to_string())
print('\nNo threshold crossings:',edge_df.first_trigger_cross_ms.isna().sum())

# crop each A/B record at its first 150 or 300 G crossing; filtering begins there.
def crop_metrics(n,threshold,fc):
 t,x,c=records[n]; xb=x-np.median(x[t>=.015],axis=0); y=xb[:,3]
 ix=np.flatnonzero(np.abs(y)>=threshold)
 if not len(ix): return np.nan,np.nan,np.nan
 k=ix[0]; z=xb[k:]; sos=signal.butter(2,fc,fs=fs,output='sos')
 fi=signal.sosfiltfilt(sos,z[:,3]); ft=np.column_stack([signal.sosfiltfilt(sos,z[:,j]) for j in range(3)])
 return np.max(abs(fi)),np.max(np.linalg.norm(ft,axis=1)),np.max(np.linalg.norm(ft,axis=1))/np.max(abs(fi))
cr=[]
for c in ['A','B']:
 for n in edge_df.loc[edge_df.config==c,'signal']:
  for th in [150,300]:
   for band,fc in [('180',300),('1000',1667)]: cr.append([n,c,th,band,*crop_metrics(n,th,fc)])
cr=pd.DataFrame(cr,columns=['signal','config','threshold','band','input','output','T'])
print('\nArtificial crop-at-threshold results')
print(cr.groupby(['config','threshold','band']).T.agg(['mean','std',lambda x:100*x.std(ddof=1)/x.mean()]).round(4).rename(columns={'<lambda_0>':'cv_pct'}).to_string())

        raw_t0                raw_first80us_mean                 raw_first80us_sd               first_trigger_cross_ms               raw_peak                     trigger          
          mean    min     max               mean     min     max             mean    min    max                   mean    min   max      mean       min       max    mean  min  max
config                                                                                                                                                                             
A        7.291  1.219  10.575              6.377  -0.204   8.382            2.426  1.005  8.057                  0.390  0.390  0.39  2049.398  1618.047  2469.913   300.0  300  300
B       13.082  3.338  24.554             15.185  12.064  21.854            5.573  3.930  6.843                  0.389  0.389  0.39   542.710   499.141   587.345   300.0  300  300
C        4.938  1.980   7.978              6.872   4.980   9.253            2.593  1.277  3.333     


Artificial crop-at-threshold results
                         mean     std  cv_pct
config threshold band                        
A      150       1000  1.1742  0.0735  6.2611
                 180   1.1600  0.0275  2.3749
       300       1000  1.1758  0.0732  6.2263
                 180   1.1087  0.0258  2.3228
B      150       1000  1.0218  0.0122  1.1935
                 180   1.0416  0.0045  0.4273
       300       1000  1.0219  0.0122  1.1937
                 180   1.0344  0.0183  1.7737


In [10]:
# Recompute with an actually available pre-trigger baseline: first 0.10 ms.
def pretrigger_metrics(n):
 t,x,c=records[n]; base=np.median(x[t<.0001],axis=0); xb=x-base; d={'signal':n,'config':c}
 for tag,fc in [('1000',1667),('180',300)]:
  sos=signal.butter(2,fc,fs=fs,output='sos'); fi=signal.sosfiltfilt(sos,xb[:,3]); ft=np.column_stack([signal.sosfiltfilt(sos,xb[:,j]) for j in range(3)])
  res=np.linalg.norm(ft,axis=1); m=t<.012; ii=np.argmax(abs(fi[m])); io=np.argmax(res[m])
  d[f'in_{tag}']=abs(fi[ii]);d[f'out_{tag}']=res[io];d[f'T_{tag}']=res[io]/abs(fi[ii])
  if tag=='180':
   d['width_ms'],d['trunc']=fwhm(t,fi,ii); v=integrate.cumulative_trapezoid(fi*9.80665,t,initial=0);d['dv_max']=max(abs(v));d['dv_final']=abs(v[-1]);d['onset_filt_frac']=abs(fi[0])/abs(fi[ii])
 d['in_raw']=max(abs(xb[:,3]));d['out_raw']=max(np.linalg.norm(xb[:,:3],axis=1));d['T_raw']=d['out_raw']/d['in_raw']
 return d
pdf=pd.DataFrame([pretrigger_metrics(n) for n in records])
rr=[]
for c,g in pdf.groupby('config'):
 z={'config':c}
 for m in ['in_raw','out_raw','T_raw','in_1000','out_1000','T_1000','in_180','out_180','T_180','width_ms','dv_max','dv_final','onset_filt_frac']:
  z[m]=g[m].mean();z[m+'_cv']=100*g[m].std(ddof=1)/g[m].mean()
 rr.append(z)
pre_sum=pd.DataFrame(rr).set_index('config')
print(pre_sum.round(4).to_string())
print('FWHM left-truncated counts',pdf.groupby('config').trunc.sum().to_dict())
# quantify raw pretrigger stability and filter acausal elevation
print('\nRaw vs filtered t=0 fractions')
for c in 'ABCD':
 ns=[n for n in records if cfg_map[n]==c]
 rawfr=[]
 for n in ns:
  t,x,_=records[n]; b=np.median(x[t<.0001],axis=0); rawfr.append(abs(x[0,3]-b[3])/max(abs(x[:,3]-b[3])))
 print(c,'raw',np.mean(rawfr),'filtered',pre_sum.loc[c,'onset_filt_frac'])

           in_raw  in_raw_cv    out_raw  out_raw_cv   T_raw  T_raw_cv   in_1000  in_1000_cv  out_1000  out_1000_cv  T_1000  T_1000_cv    in_180  in_180_cv   out_180  out_180_cv   T_180  T_180_cv  width_ms  width_ms_cv  dv_max  dv_max_cv  dv_final  dv_final_cv  onset_filt_frac  onset_filt_frac_cv
config                                                                                                                                                                                                                                                                                                  
A       2042.7088    14.5365  2097.3777      7.3968  1.0382    9.3386  806.4292      4.9832  944.4175       1.8958  1.1745     6.4122  363.4589     1.7292  376.9977      1.2296  1.0373    0.5376    1.6444       1.4762  5.4286     0.6332    3.9652       6.1913           0.5206              1.5695
B        526.0283     5.4328   409.2385      7.6028  0.7783    6.3978  326.7396      4.5799  340.4623       5

In [11]:
# SRS theory for a unit half-sine base-acceleration pulse: relative-motion SDOF, absolute acceleration maximax.
from scipy.integrate import solve_ivp

def halfsine_srs(r,zeta=.05): # r=f_n*tau; tau=1, input peak=1
    wn=2*np.pi*r
    def fun(t,q):
        a=np.sin(np.pi*t) if 0<=t<=1 else 0.0
        return [q[1], -2*zeta*wn*q[1]-wn**2*q[0]-a]
    horizon=max(5,8/(zeta*wn) if zeta>0 else 10/r)
    tt=np.linspace(0,min(horizon,100),max(5000,int(min(horizon,100)*3000)))
    sol=solve_ivp(fun,[0,tt[-1]],[0,0],t_eval=tt,rtol=1e-8,atol=1e-10)
    a_in=np.where(tt<=1,np.sin(np.pi*tt),0)
    absacc=-2*zeta*wn*sol.y[1]-wn**2*sol.y[0] # = relative acc + base acc
    return np.max(np.abs(absacc))
rs=np.logspace(-2,1,120)
for z in [0.01,.05,.1,.2]:
 vals=np.array([halfsine_srs(r,z) for r in rs]); im=vals.argmax()
 print('zeta',z,'peak r',rs[im],'amp',vals[im], 'at r .91,1.24,1.84', [round(halfsine_srs(r,z),3) for r in [.91,1.24,1.84]])
# local log-sensitivity magnitude d ln SRS / d ln r
for z in [.01,.05,.1,.2]:
 for r in [.91,1.24,1.84]:
  e=.01; sens=(np.log(halfsine_srs(r*(1+e),z))-np.log(halfsine_srs(r*(1-e),z)))/(np.log(r*(1+e))-np.log(r*(1-e)))
  print(z,r,'S',halfsine_srs(r,z),'logslope',sens)

zeta 0.01 peak r 0.8240743309893661 amp 1.7426934345278327 at r .91,1.24,1.84 [np.float64(1.732), np.float64(1.609), np.float64(1.324)]


zeta 0.05 peak r 0.8240743309893661 amp 1.6516282806669056 at r .91,1.24,1.84 [np.float64(1.644), np.float64(1.536), np.float64(1.275)]


zeta 0.1 peak r 0.8240743309893661 amp 1.5583349563736344 at r .91,1.24,1.84 [np.float64(1.553), np.float64(1.46), np.float64(1.227)]


zeta 0.2 peak r 0.8733261623828433 amp 1.4210998938185544 at r .91,1.24,1.84 [np.float64(1.419), np.float64(1.349), np.float64(1.163)]


0.01 0.91 S 1.7323953250550943 logslope -0.10557715742555517


...

0.05 0.91 S 1.6440926347980616 logslope -0.09063240363782796


0.05 1.24 S 1.5359019162748442 logslope -0.34122173173874215


0.05 1.84 S 1.2753539737779667 logslope -0.586559006062277
0.1 0.91 S 1.553336652431463 logslope -0.0750400303530578


0.1 1.24 S 1.4604951021609434 logslope -0.3156746477481891
0.1 1.84 S 1.227437854950751 logslope -0.5512985165396956


0.2 0.91 S 1.4189671803386632 logslope -0.05221384406346313
0.2 1.24 S 1.3489453416485635 logslope -0.26780401647433283
0.2 1.84 S 1.16263953382259 logslope -0.4687035687434064


In [12]:
# Temporal dependence, trends, variance uncertainty, and power.
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm
from scipy.stats import chi2, f as fdist
statrows=[]
for metric in ['T_180','T_1000','in_raw','in_180','width_ms']:
 for c,g in df.groupby('config'):
  y=g.sort_values('signal')[metric].to_numpy(); x=np.arange(1,11)
  fit=stats.linregress(x,y); resid=y-(fit.intercept+fit.slope*x)
  r1=np.corrcoef(y[:-1],y[1:])[0,1]; rr1=np.corrcoef(resid[:-1],resid[1:])[0,1]
  neff=10*(1-r1)/(1+r1) if abs(r1)<1 else np.nan; neff=max(1,min(10,neff))
  statrows.append([metric,c,fit.slope,100*fit.slope/y.mean(),fit.pvalue,fit.rvalue**2,r1,rr1,neff,durbin_watson(resid)])
statsdf=pd.DataFrame(statrows,columns=['metric','config','slope','slope_pct_drop','trend_p','R2','lag1_raw','lag1_resid','n_eff_AR1','DW_resid'])
print(statsdf.round(4).to_string(index=False))
# exact-ish variance ratio A/B and bootstrap CIs for CV via iid and moving-block bootstrap (descriptive only)
rng=np.random.default_rng(1234)
def boot_cv(y,B=20000,block=None):
 n=len(y); vals=[]
 for _ in range(B):
  if block is None: z=rng.choice(y,n,replace=True)
  else:
   starts=rng.integers(0,n-block+1,size=int(np.ceil(n/block))); z=np.concatenate([y[s:s+block] for s in starts])[:n]
  vals.append(100*np.std(z,ddof=1)/np.mean(z))
 return np.percentile(vals,[2.5,50,97.5])
for m in ['T_180','T_1000']:
 print('\n',m)
 for c in 'ABCD':
  y=df[df.config==c][m].values
  print(c,'CV',100*y.std(ddof=1)/y.mean(),'iidCI',boot_cv(y,5000),'block2CI',boot_cv(y,5000,2))
 a=df[df.config=='A'][m].values;b=df[df.config=='B'][m].values
 vr=np.var(a,ddof=1)/np.var(b,ddof=1); p2=2*min(fdist.cdf(vr,9,9),1-fdist.cdf(vr,9,9)); print('A/B variance ratio',vr,'naive F p',p2,'Levene',stats.levene(a,b,center='median'))
# C-D detectable difference under iid Welch, empirical sd, alpha .05 power .8
from statsmodels.stats.power import TTestIndPower
for m in ['T_180','T_1000']:
 g1=df[df.config=='C'][m];g2=df[df.config=='D'][m]; sp=np.sqrt((g1.var(ddof=1)+g2.var(ddof=1))/2)
 dcrit=TTestIndPower().solve_power(nobs1=10,alpha=.05,power=.8,ratio=1,alternative='two-sided'); print('C/D',m,'obs diff pct',100*(g2.mean()-g1.mean())/g1.mean(),'MDE pct',100*dcrit*sp/((g1.mean()+g2.mean())/2),'dcrit',dcrit)

  metric config   slope  slope_pct_drop  trend_p     R2  lag1_raw  lag1_resid  n_eff_AR1  DW_resid
   T_180      A  0.0001          0.0125   0.8168 0.0071   -0.3237     -0.3896    10.0000    2.0510
   T_180      B  0.0004          0.0378   0.2327 0.1724   -0.0505     -0.1724    10.0000    2.1105
   T_180      C  0.0026          0.2488   0.0156 0.5390    0.8225      0.5877     1.0000    0.7458
   T_180      D -0.0012         -0.1110   0.0299 0.4648    0.7141      0.5470     1.6681    0.8258
  T_1000      A  0.0225          1.9098   0.0000 0.8899    0.9823      0.3805     1.0000    0.8284
  T_1000      B -0.0027         -0.2614   0.0366 0.4398    0.3998     -0.0744     4.2877    1.6151
  T_1000      C  0.0005          0.0468   0.6652 0.0246    0.2425      0.2439     6.0962    1.4472
  T_1000      D -0.0032         -0.2834   0.0202 0.5106    0.5539      0.3024     2.8707    1.3473
  in_raw      A 94.7534          4.6235   0.0000 0.9286    0.9316      0.0826     1.0000    1.6235
  in_raw  

A CV 0.44703214624113197 iidCI [0.19188206 0.42135139 0.57555333] block2CI [0.13479543 0.39173498 0.59317435]


B CV 0.2756500723242605 iidCI [0.12757317 0.26399042 0.34021433] block2CI [0.11662168 0.2557281  0.33513109]


C CV 1.0261802542379361 iidCI [0.5437203  0.98045184 1.22327041] block2CI [0.34721316 0.95859308 1.24742457]


D CV 0.49292881596183435 iidCI [0.18700277 0.46217006 0.64731437] block2CI [0.15752504 0.37372082 0.59577773]
A/B variance ratio 2.6668949382307057 naive F p 0.160110272717785 Levene LeveneResult(statistic=np.float64(0.781433432977476), pvalue=np.float64(0.38835820228034973))

 T_1000


A CV 6.129633393948425 iidCI [2.64653233 5.84851982 8.24473537] block2CI [1.55385268 4.49369142 7.5253199 ]


B CV 1.1934229383977188 iidCI [0.38137712 1.11565329 1.64344594] block2CI [0.32864702 0.94605098 1.49351603]


C CV 0.902705023558933 iidCI [0.42395866 0.85080614 1.16238261] block2CI [0.45738436 0.86619751 1.14135015]


D CV 1.200588236624626 iidCI [0.61539424 1.12991606 1.5342786 ] block2CI [0.53065686 0.99830822 1.42442567]
A/B variance ratio 35.06961314246792 naive F p 1.2024619558292926e-05 Levene LeveneResult(statistic=np.float64(8.707355415230083), pvalue=np.float64(0.008551384436586143))
C/D T_180 obs diff pct -0.37839778977977073 MDE pct 1.0678370860307027 dcrit 1.324947368330822
C/D T_1000 obs diff pct -0.5839655368085593 MDE pct 1.4061426019175933 dcrit 1.324947368330822


In [13]:
# Trend-adjusted residual variability and block-level identifiability.
trendvar=[]
for m in ['T_180','T_1000']:
 for c,g in df.groupby('config'):
  y=g[m].values;x=np.arange(10);fit=stats.linregress(x,y);res=y-(fit.intercept+fit.slope*x)
  trendvar.append([m,c,y.mean(),100*y.std(ddof=1)/y.mean(),100*res.std(ddof=1)/y.mean(),fit.slope,fit.pvalue])
print(pd.DataFrame(trendvar,columns=['metric','config','mean','raw_cv','detrended_resid_cv','slope','p']).round(5).to_string(index=False))
# Pairwise drop-level tests contrasted with the design-valid statement (one block each).
for m in ['T_180','T_1000']:
 print('\n',m)
 for i,a in enumerate('ABCD'):
  for b in 'ABCD'[i+1:]:
   x=df[df.config==a][m];y=df[df.config==b][m];tt=stats.ttest_ind(x,y,equal_var=False)
   print(a,b,'means',x.mean(),y.mean(),'p_naive',tt.pvalue)
# Monotonicity based on four block means and duration
bm=df.groupby('config')[['width_ms','T_180','T_1000']].mean()
print('\nblock means\n',bm)
print('Spearman width-T180 four arrangements',stats.spearmanr(bm.width_ms,bm.T_180))

metric config    mean  raw_cv  detrended_resid_cv    slope       p
 T_180      A 1.04460 0.44703             0.44544  0.00013 0.81676
 T_180      B 1.03735 0.27565             0.25076  0.00039 0.23273
 T_180      C 1.04640 1.02618             0.69676  0.00260 0.01562
 T_180      D 1.04244 0.49293             0.36061 -0.00116 0.02990
T_1000      A 1.17818 6.12963             2.03401  0.02250 0.00004
T_1000      B 1.02185 1.19342             0.89323 -0.00267 0.03659
T_1000      C 1.13163 0.90271             0.89153  0.00053 0.66516
T_1000      D 1.12502 1.20059             0.83990 -0.00319 0.02023

 T_180
A B means 1.0445973822052386 1.037353134030815 p_naive 0.0008077542760799217
A C means 1.0445973822052386 1.046401741022475 p_naive 0.6346304271341495
A D means 1.0445973822052386 1.042442179962229 p_naive 0.33944112055276404
B C means 1.037353134030815 1.046401741022475 p_naive 0.027117012858040464
B D means 1.037353134030815 1.042442179962229 p_naive 0.015989063106058812
C D means 1.0

In [14]:
from e14c.web import web_search
queries=[
 'site:sae.org J211 instrumentation impact test channel frequency class filtering CFC',
 'site:iso.org ISO 16063-21 accelerometer mounting mass loading vibration calibration',
 'site:iso.org ISO 5347 accelerometer mounting',
 'site:quicksearch.dla.mil MIL-STD-810H Method 516.8 shock response spectrum',
 'site:astm.org D3332 mechanical-shock fragility products',
 'site:astm.org D7136 drop-weight impact damage fiber reinforced polymer matrix composite',
 'IEST RP DTE012 shock response spectrum testing'
]
search_results={}
for q in queries:
    search_results[q]=await web_search(q,num_results=5)
    print('\nQUERY',q)
    for r in search_results[q][:3]: print(r.get('title'),r.get('url'),(r.get('snippet') or '')[:300])


QUERY site:sae.org J211 instrumentation impact test channel frequency class filtering CFC
J211/1_202208: Instrumentation for Impact Test Part 1 - Electronic Instrumentation - Recommended Practice https://saemobilus.sae.org/standards/j2111_202208-instrumentation-impact-test-part-1-electronic-instrumentation J211/1_202208: Instrumentation for Impact Test Part 1 - Electronic Instrumentation - Recommended Practice
...
This SAE Recommended Practice outlines a series of performance recommendations, which concern the whole data channel. These recommendations are not subject to any variation and all of them s
2002-01-0796: Digital Filtering for J211 Requirements using a Fast Fourier Transform Based Filter - Technical Paper https://saemobilus.sae.org/papers/digital-filtering-j211-requirements-using-a-fast-fourier-transform-based-filter-2002-01-0796 The need for low pass filters stems from a need to eliminate high frequency noise from raw data (the output of the data acquisition system). As an 


QUERY site:iso.org ISO 16063-21 accelerometer mounting mass loading vibration calibration
ISO 16063-21:2003 - Methods for the calibration of vibration and shock transducers — Part 21: Vibration calibration by comparison to a reference transducer https://www.iso.org/standard/27053.html ISO 16063-21:2003 - Methods for the calibration of vibration and shock transducers — Part 21: Vibration calibration by comparison to a reference transducer
...
Methods for the calibration of vibration and shock transducers — Part 21: Vibration calibration by comparison to a reference transducer
...

ISO 16063-21:2003/Amd 2:2024 - Methods for the calibration of vibration and shock transducers — Part 21: Vibration calibration by comparison to a reference transducer — Amendment 2 https://www.iso.org/standard/85624.html ISO 16063-21:2003/Amd 2:2024 - Methods for the calibration of vibration and shock transducers — Part 21: Vibration calibration by comparison to a reference transducer — Amendment 2
...
Method


QUERY site:iso.org ISO 5347 accelerometer mounting
ISO 5348:2021 - Mechanical vibration and shock — Mechanical mounting of accelerometers https://www.iso.org/standard/78160.html ISO 5348:2021 - Mechanical vibration and shock — Mechanical mounting of accelerometers
...
Mechanical vibration and shock — Mechanical mounting of accelerometers
...
This document specifies the important technical properties of the different methods for mounting vibration transducers and describes r
ISO 5348:1987 - Mechanical vibration and shock — Mechanical mounting of accelerometers https://www.iso.org/standard/11368.html ISO 5348:1987 - Mechanical vibration and shock — Mechanical mounting of accelerometers
...
Mechanical vibration and shock — Mechanical mounting of accelerometers
ISO 5348:1998(en), Mechanical vibration and shock https://www.iso.org/obp/ui#iso:std:iso:5348:ed-2:en ISO/IEC/IEEE 29119-5:2024(en), Software and systems engineering — Software testing — Part 5: Keyword-driven testing
...
ISO/IEC/I


QUERY site:quicksearch.dla.mil MIL-STD-810H Method 516.8 shock response spectrum
ASSIST-QuickSearch Document Details https://quicksearch.dla.mil/qsDocDetails.aspx?ident_number=35978 | Document ID: | MIL-STD-810 Scroll down to access document images |
| --- | --- |
...
| Title: | Environmental Engineering Considerations and Laboratory Tests |
| --- | --- |
| Scope: | This Standard contains materiel acquisition program planning and engineering direction for considering the influe
 https://quicksearch.dla.mil/WMX/Default.aspx?token=41004 MIL–STD–810E has been revised to require careful attention to environments throughout the development
...
process. A course of action for determining and assessing the environments to which an item will be exposed
...
The bulk of the standard remains devoted to test methods. Individual methods have 
 https://quicksearch.dla.mil/Transient/7C2ECEBC5BBC4CE1B3EE525E47B983A6.pdf MIL-STD-810 - Environmental Engineering Considerations and Laboratory Tests



QUERY site:astm.org D3332 mechanical-shock fragility products
D3332 Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines https://store.astm.org/d3332-99r23.html D3332 Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines
...
# Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines
...
Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines D3332-99R23 ASTM|D3332-
D3332 Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines https://store.astm.org/d3332-99r16.html D3332 Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines
...
# Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines
...
Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines D3332-99R16 ASTM|D3332-
D3332 Standard Test Methods for Mechanical-Shock ... https://store.as


QUERY site:astm.org D7136 drop-weight impact damage fiber reinforced polymer matrix composite
D7136/D7136M Standard Test Method for  Measuring the Damage Resistance of a Fiber-Reinforced Polymer  Matrix Composite to a Drop-Weight Impact Event https://store.astm.org/d7136_d7136m-25.html D7136/D7136M Standard Test Method for Measuring the Damage Resistance of a Fiber-Reinforced Polymer Matrix Composite to a Drop-Weight Impact Event
...
# Standard Test Method for Measuring the Damage Resistance of a Fiber-Reinforced Polymer Matrix Composite to a Drop-Weight Impact Event
...
Standard 
D7136/D7136M Standard Test Method for  Measuring the Damage Resistance of a Fiber-Reinforced Polymer  Matrix Composite to a Drop-Weight Impact Event https://store.astm.org/d7136_d7136m-20.html D7136/D7136M Standard Test Method for Measuring the Damage Resistance of a Fiber-Reinforced Polymer Matrix Composite to a Drop-Weight Impact Event
...
# Standard Test Method for Measuring the Damage Resistance of a Fib


QUERY IEST RP DTE012 shock response spectrum testing
Shock Response Spectrum (SRS): The Comprehensive ... https://caeflow.com/vibration_and_acoustics/shock-response-spectrum-srs/ # Shock Response Spectrum (SRS): The Comprehensive Engineering Guide
...
- Definition: SRS plots the maximum response of a series of single-degree-of-freedom (SDOF) oscillators to a transient event.
- Key Application: Used for qualifying hardware against pyroshock, seismic events, and drop impacts.

IEST-RD-DTE012: Handbook for Dynamic Data Acquisition ... https://www.iest.org/Standards-RPs/Recommended-Practices/IEST-RD-DTE012 IEST-RD-DTE012: Handbook for Dynamic Data Acquisition and Analysis
...
The purpose of this Handbook, IEST-RD-DTE012.2, is to provide guidelines for acquiring and analyzing structural (or mechanical) shock and vibration, and acoustic and aerodynamic noise data from flight and ground tests for all cat
The Shock Response Spectrum – A Primer https://sem-proceedings.com/27i/sem.org-IMAC-XXVI

In [15]:
# Causal-context reconstruction: prepend a flat baseline before filtering, then compare peaks.
def prepend_variant(n,fc,pre_ms):
 t,x,c=records[n]; b=np.median(x[t<.0001],axis=0); xb=x-b
 k=int(pre_ms*1e-3*fs); z=np.vstack([np.zeros((k,4)),xb]); sos=signal.butter(2,fc,fs=fs,output='sos')
 fi=signal.sosfiltfilt(sos,z[:,3])[k:]; ft=np.column_stack([signal.sosfiltfilt(sos,z[:,j])[k:] for j in range(3)])
 m=t<.012; pi=max(abs(fi[m]));po=max(np.linalg.norm(ft[m],axis=1)); return pi,po,po/pi,abs(fi[0])/pi
pr=[]
for band,fc in [('180',300),('1000',1667)]:
 for pre in [0,1,2,5,10]:
  for n in records:
   pr.append([n,cfg_map[n],band,pre,*prepend_variant(n,fc,pre)])
prd=pd.DataFrame(pr,columns=['signal','config','band','prepend_ms','input','output','T','onset'])
q=prd.groupby(['band','config','prepend_ms']).agg(Tmean=('T','mean'),Tcv=('T',lambda x:100*x.std(ddof=1)/x.mean()),input=('input','mean'),output=('output','mean'),onset=('onset','mean')).reset_index()
print(q.round(4).to_string(index=False))
print('\nRelative difference 0 vs 10ms T means')
for band in ['180','1000']:
 for c in 'ABCD':
  a=q[(q.band==band)&(q.config==c)].set_index('prepend_ms').Tmean
  print(band,c,100*(a[0]-a[10])/a[10])

band config  prepend_ms  Tmean    Tcv    input   output  onset
1000      A           0 1.1745 6.4122 806.4292 944.4175 0.0220
1000      A           1 1.1745 6.4125 806.4292 944.4174 0.0224
1000      A           2 1.1745 6.4125 806.4292 944.4174 0.0224
1000      A           5 1.1745 6.4125 806.4292 944.4174 0.0224
1000      A          10 1.1745 6.4125 806.4292 944.4174 0.0224
1000      B           0 1.0417 1.1134 326.7396 340.4623 0.0206
1000      B           1 1.0417 1.1134 326.7396 340.4623 0.0126
1000      B           2 1.0417 1.1134 326.7396 340.4623 0.0126
1000      B           5 1.0417 1.1134 326.7396 340.4623 0.0126
1000      B          10 1.0417 1.1134 326.7396 340.4623 0.0126
1000      C           0 1.1355 1.5389 175.5577 199.3405 0.0138
1000      C           1 1.1355 1.5389 175.5577 199.3405 0.0061
1000      C           2 1.1355 1.5389 175.5577 199.3405 0.0061
1000      C           5 1.1355 1.5389 175.5577 199.3405 0.0061
1000      C          10 1.1355 1.5389 175.5577 199.3405

In [16]:
# Ringdown/modal diagnostics by physical axis, using 4-18 ms to exclude primary pulse.
modal=[]
for n,(t,x,c) in records.items():
 b=np.median(x[t<.0001],axis=0); xb=x-b
 m=(t>=.004)&(t<=.018); seg=signal.detrend(xb[m],axis=0); w=signal.windows.tukey(seg.shape[0],.2)
 # zero padding changes interpolation, not true 71 Hz resolution
 N=131072; X=np.fft.rfft(seg*w[:,None],n=N,axis=0); fz=np.fft.rfftfreq(N,1/fs); band=(fz>=400)&(fz<=900); P=abs(X)**2
 row={'signal':n,'config':c}
 for j,ch in enumerate(['CH2','CH3','CH4','CH5']):
  k=np.argmax(P[band,j]); row[ch+'_peak']=fz[band][k];row[ch+'_bandE']=P[band,j].sum()
 modal.append(row)
mdf=pd.DataFrame(modal)
print(mdf.groupby('config')[[c for c in mdf if 'peak' in c]].agg(['mean','std','min','max']).round(1).to_string())
# axis energy shares, peak frequency correlations with raw severity/drop order
for c,g in mdf.groupby('config'):
 print('\n',c)
 for ch in ['CH2','CH3','CH4']:
  print(ch,'energy share', (g[ch+'_bandE']/g[[a+'_bandE' for a in ['CH2','CH3','CH4']]].sum(axis=1)).mean())
 merged=g.merge(pdf[['signal','in_raw']])
 for ch in ['CH2','CH3','CH4']:
  print(ch,'peak vs severity rho',stats.spearmanr(merged[ch+'_peak'],merged.in_raw))
# dominant axis identities and input-output peak equality within interpolation tolerance 20 Hz
print('\ndominant output axis counts')
for c,g in mdf.groupby('config'):
 dom=g[[a+'_bandE' for a in ['CH2','CH3','CH4']]].idxmax(axis=1).str[:3]
 print(c,dom.value_counts().to_dict(),'CH5/output dominant freq mean absolute gaps',np.mean(abs(g.CH5_peak-g[[a+'_bandE' for a in ['CH2','CH3','CH4']]].idxmax(axis=1).map(lambda col: 0))))

       CH2_peak                     CH3_peak                     CH4_peak                     CH5_peak                     
           mean   std    min    max     mean   std    min    max     mean   std    min    max     mean    std    min    max
config                                                                                                                     
A         536.9   7.9  524.5  543.6    516.9   6.0  505.4  524.5    549.3  58.5  524.5  715.3    815.4  171.0  476.8  896.5
B         410.1  30.2  400.5  495.9    533.1   3.0  524.5  534.1    430.1  62.4  400.5  553.1    496.9   49.7  400.5  543.6
C         400.5   0.0  400.5  400.5    511.2  36.9  457.8  534.1    400.5   0.0  400.5  400.5    416.8    4.6  410.1  419.6
D         400.5   0.0  400.5  400.5    476.8  53.9  400.5  534.1    400.5   0.0  400.5  400.5    441.6   33.4  410.1  486.4

 A
CH2 energy share 0.31718810878416975
CH3 energy share 0.5096434194505783
CH4 energy share 0.17316847176525194
CH2 peak vs severi

/tmp/ipykernel_20/2164043279.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  print(ch,'peak vs severity rho',stats.spearmanr(merged[ch+'_peak'],merged.in_raw))
/tmp/ipykernel_20/2164043279.py:21: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  print(ch,'peak vs severity rho',stats.spearmanr(merged[ch+'_peak'],merged.in_raw))


In [17]:
# Spectral-estimator sensitivity: reproduce published nonlinear-resultant Welch and vary estimator/window.
spectral_sens=[]
for n,(t,x,c) in records.items():
 xb=x-np.median(x,axis=0)
 # impact from published-like CFC180
 sos=signal.butter(2,300.06,fs=fs,output='sos'); fi=signal.sosfiltfilt(sos,xb[:,3]); ii=np.argmax(abs(fi[:15000]))
 lo=max(0,ii-int(.001*fs));hi=min(len(t),ii+int(.008*fs)); rawres=np.linalg.norm(xb[:,:3],axis=1)
 seg=rawres[lo:hi]; fw,pw=signal.welch(seg-seg.mean(),fs=fs,nperseg=min(4096,len(seg)))
 den=(fw>=100)&(fw<=20000);bn=(fw>=450)&(fw<=800)
 published_like=pw[bn].sum()/pw[den].sum()
 # full 20ms, sum of linear axis periodograms, common tapers
 for winname,w in [('rect',np.ones(len(t))),('hann',signal.windows.hann(len(t))),('tukey20',signal.windows.tukey(len(t),.2))]:
  z=signal.detrend((x-np.median(x[t<.0001],axis=0))[:,:3],axis=0)*w[:,None]
  X=np.fft.rfft(z,axis=0);f=np.fft.rfftfreq(len(t),1/fs);P=(abs(X)**2).sum(axis=1); den2=(f>=100)&(f<=20000);bn2=(f>=450)&(f<=800)
  spectral_sens.append([n,c,published_like,winname,P[bn2].sum()/P[den2].sum()])
ssd=pd.DataFrame(spectral_sens,columns=['signal','config','published_like','window','linear_axis_share'])
print(ssd.groupby('config').published_like.agg(['mean','std']).round(4))
print('\nLinear axis sums, full record')
print(ssd.groupby(['window','config']).linear_axis_share.agg(['mean','std']).round(4))
print('\nOriginal Welch bins in band, frequencies:',fw[bn], 'resolution',fw[1]-fw[0], 'segment A begins at zero count',sum(max(0,np.argmax(abs(signal.sosfiltfilt(signal.butter(2,300.06,fs=fs,output="sos"),(records[n][1][:,3]-np.median(records[n][1][:,3])))[:15000]))-int(.001*fs))==0 for n in records if cfg_map[n]=='A'))

          mean     std
config                
A       0.1662  0.0040
B       0.2250  0.0143
C       0.1079  0.0041
D       0.1300  0.0058

Linear axis sums, full record
                  mean     std
window  config                
hann    A       0.3606  0.0484
        B       0.1713  0.0143
        C       0.0229  0.0043
        D       0.0143  0.0040
rect    A       0.2205  0.0281
        B       0.0360  0.0068
        C       0.0274  0.0020
        D       0.0360  0.0021
tukey20 A       0.3065  0.0341
        B       0.0695  0.0105
        C       0.0140  0.0018
        D       0.0146  0.0012

Original Welch bins in band, frequencies: [610.3515625] resolution 305.1757812499926 segment A begins at zero count 10


In [18]:
s=await check_task(standards_task)
print('status',s.status)
print((s.answer or '')[:12000])

Task 74990183-f4d6-4b25-8d07-dd3a48f338d6 still running (status: in progress). Call check_task(task_id) again later.
status in progress



In [19]:
# Rebuild and export the final per-drop and arrangement tables.
final_rows=[]
for n,(t,x,c) in records.items():
    b=np.median(x[t<.0001],axis=0); xb=x-b; r={'signal':n,'arrangement':c}
    for tag,fc in [('raw',None),('CFC1000',1667),('CFC180',300)]:
        if fc is None:
            fi=xb[:,3]; ft=xb[:,:3]
        else:
            sos=signal.butter(2,fc,fs=fs,output='sos')
            fi=signal.sosfiltfilt(sos,xb[:,3])
            ft=np.column_stack([signal.sosfiltfilt(sos,xb[:,j]) for j in range(3)])
        res=np.linalg.norm(ft,axis=1); m=t<.012
        r[f'input_{tag}_G']=np.max(abs(fi[m])); r[f'output_{tag}_G']=np.max(res[m]); r[f'T_{tag}']=r[f'output_{tag}_G']/r[f'input_{tag}_G']
        if tag=='CFC180':
            ii=np.argmax(abs(fi[m])); r['pulse_FWHM_ms'],r['FWHM_left_truncated']=fwhm(t,fi,ii)
            v=integrate.cumulative_trapezoid(fi[m]*9.80665,t[m],initial=0)
            r['captured_delta_v_0_12ms_m_s']=np.max(abs(v)); r['CFC180_t0_fraction']=abs(fi[0])/r[f'input_{tag}_G']
    final_rows.append(r)
final_df=pd.DataFrame(final_rows).sort_values('signal')
metric_cols=[c for c in final_df if c not in ['signal','arrangement','FWHM_left_truncated']]
arr_rows=[]
for c,g in final_df.groupby('arrangement'):
    for m in metric_cols:
        mean=g[m].mean(); sd=g[m].std(ddof=1); sem=sd/np.sqrt(len(g)); ci=stats.t.ppf(.975,len(g)-1)*sem
        arr_rows.append({'arrangement':c,'metric':m,'n':len(g),'mean':mean,'sd':sd,'CV_pct':100*sd/mean,'mean_CI95_low':mean-ci,'mean_CI95_high':mean+ci})
arr_long=pd.DataFrame(arr_rows)
outdir=Path('/workspace/edison-trajectories/pu-configs');outdir.mkdir(parents=True,exist_ok=True)
final_df.to_csv(outdir/'independent_per_drop_metrics.csv',index=False)
arr_long.to_csv(outdir/'independent_arrangement_summary.csv',index=False)
print(final_df.round(4).to_string(index=False))
print('\nArrangement table')
wide=final_df.groupby('arrangement')[metric_cols].agg(['mean','std'])
for m in metric_cols: wide[(m,'CV_pct')]=100*wide[(m,'std')]/wide[(m,'mean')]
print(wide.round(4).to_string())

 signal arrangement  input_raw_G  output_raw_G  T_raw  input_CFC1000_G  output_CFC1000_G  T_CFC1000  input_CFC180_G  output_CFC180_G  T_CFC180  pulse_FWHM_ms  FWHM_left_truncated  captured_delta_v_0_12ms_m_s  CFC180_t0_fraction
      1           A    1614.8702     1725.5737 1.0686         892.9573          904.0317     1.0124        376.9817         386.6355    1.0256         1.5934                 True                       5.5014              0.5380
      2           A    1658.9467     1941.5464 1.1703         843.7305          927.2643     1.0990        369.1827         380.8103    1.0315         1.6212                 True                       5.4525              0.5294
      3           A    1793.1712     2075.7428 1.1576         824.6111          931.5586     1.1297        365.8669         380.1169    1.0389         1.6314                 True                       5.4221              0.5200
      4           A    1940.1467     2153.6672 1.1101         811.0858          950.9242

In [20]:
# Correlation diagnostics for A's broadband instability and arrangement summary CIs used in report.
for c in 'ABCD':
 g=final_df[final_df.arrangement==c].merge(edge_df[['signal','raw_peak']],on='signal')
 print('\n',c)
 for m in ['T_CFC1000','T_CFC180']:
  print(m,'vs drop',stats.spearmanr(g.signal,g[m]),'vs raw peak',stats.spearmanr(g.raw_peak,g[m]))
# Selected arrangement estimates with 95% t CIs
sel=['input_raw_G','output_raw_G','input_CFC1000_G','output_CFC1000_G','T_CFC1000','input_CFC180_G','output_CFC180_G','T_CFC180','pulse_FWHM_ms','captured_delta_v_0_12ms_m_s']
for c,g in final_df.groupby('arrangement'):
 print('\n',c)
 for m in sel:
  ci=stats.t.interval(.95,len(g)-1,loc=g[m].mean(),scale=stats.sem(g[m]))
  print(m,round(g[m].mean(),4),tuple(round(v,4) for v in ci),'CV',round(100*g[m].std(ddof=1)/g[m].mean(),3))


 A
T_CFC1000 vs drop SignificanceResult(statistic=np.float64(0.9999999999999999), pvalue=np.float64(6.64689742203186e-64)) vs raw peak SignificanceResult(statistic=np.float64(0.9636363636363635), pvalue=np.float64(7.3209748095299185e-06))
T_CFC180 vs drop SignificanceResult(statistic=np.float64(0.8181818181818182), pvalue=np.float64(0.003814920082550713)) vs raw peak SignificanceResult(statistic=np.float64(0.7333333333333332), pvalue=np.float64(0.015800596250571577))

 B
T_CFC1000 vs drop SignificanceResult(statistic=np.float64(-0.6484848484848483), pvalue=np.float64(0.04254012768448949)) vs raw peak SignificanceResult(statistic=np.float64(0.6242424242424242), pvalue=np.float64(0.053717767217167436))
T_CFC180 vs drop SignificanceResult(statistic=np.float64(0.006060606060606061), pvalue=np.float64(0.9867429111949892)) vs raw peak SignificanceResult(statistic=np.float64(0.3939393939393939), pvalue=np.float64(0.25999776683488757))

 C
T_CFC1000 vs drop SignificanceResult(statistic=np.flo

In [21]:
s=await check_task(standards_task)
print('status',s.status)
print((s.answer or '')[:20000])

Task 74990183-f4d6-4b25-8d07-dd3a48f338d6 still running (status: in progress). Call check_task(task_id) again later.
status in progress



In [22]:
# Sensitivity of the corrected baseline to the usable raw pre-trigger window length.
base_sens=[]
for pre_us in [50,100,200,300]:
 for n,(t,x,c) in records.items():
  b=np.median(x[t<pre_us*1e-6],axis=0);xb=x-b
  for tag,fc in [('180',300),('1000',1667)]:
   sos=signal.butter(2,fc,fs=fs,output='sos');fi=signal.sosfiltfilt(sos,xb[:,3]);ft=np.column_stack([signal.sosfiltfilt(sos,xb[:,j]) for j in range(3)])
   m=t<.012;tr=np.max(np.linalg.norm(ft[m],axis=1))/np.max(abs(fi[m]));base_sens.append([pre_us,n,c,tag,tr])
bss=pd.DataFrame(base_sens,columns=['pre_us','signal','config','band','T'])
print(bss.groupby(['band','config','pre_us']).T.agg(['mean',lambda x:100*x.std(ddof=1)/x.mean()]).round(4).rename(columns={'<lambda_0>':'CV_pct'}).to_string())
print('\nRaw CH5 mean by 50us segment before threshold crossing (G relative to first 100us median):')
for c in 'ABCD':
 vals=[]
 for n in [n for n in records if cfg_map[n]==c]:
  t,x,_=records[n];b=np.median(x[t<.0001,3]);vals.append([np.mean(x[(t>=a)&(t<a+50e-6),3]-b) for a in np.arange(0,.00035,50e-6)])
 print(c,np.mean(vals,axis=0).round(2))

                      mean  CV_pct
band config pre_us                
1000 A      50      1.1751  6.4706
            100     1.1745  6.4122
            200     1.1766  6.3747
            300     1.1795  6.2922
     B      50      1.0355  1.9083
            100     1.0417  1.1134
            200     1.0511  1.7014
            300     1.0590  2.0400
     C      50      1.1306  0.9811
            100     1.1355  1.5389
            200     1.1627  2.3713
            300     1.2349  4.2457
     D      50      1.1600  2.7996
            100     1.1792  3.0158
            200     1.2135  1.5023
            300     1.2296  0.9424
180  A      50      1.0385  0.6883
            100     1.0373  0.5376
            200     1.0415  0.4569
            300     1.0474  0.3108
     B      50      1.0549  1.8471
            100     1.0633  1.1784
            200     1.0763  1.5460
            300     1.0874  2.1057
     C      50      1.0450  0.9430
            100     1.0497  1.2232
            200     

In [23]:
# Generate compact markdown rows for the report and a published-vs-corrected comparison.
def compact_table(d):
 lines=['|Sig.|Cfg.|Raw in/out (G)|T raw|CFC1000 in/out (G)|T1000|CFC180 in/out (G)|T180|FWHM (ms)|Captured Δv (m/s)|','|---:|:--:|---:|---:|---:|---:|---:|---:|---:|---:|']
 for _,r in d.iterrows():
  lines.append(f"|{int(r.signal)}|{r.arrangement}|{r.input_raw_G:.0f}/{r.output_raw_G:.0f}|{r.T_raw:.3f}|{r.input_CFC1000_G:.1f}/{r.output_CFC1000_G:.1f}|{r.T_CFC1000:.3f}|{r.input_CFC180_G:.1f}/{r.output_CFC180_G:.1f}|{r.T_CFC180:.3f}|{r.pulse_FWHM_ms:.3f}{'*' if r.FWHM_left_truncated else ''}|{r.captured_delta_v_0_12ms_m_s:.3f}|")
 return '\n'.join(lines)
perdrop_md=compact_table(final_df)
print(perdrop_md[:5000])
# Published comparison chosen metrics from JSON and corrected values.
for c,key in zip('ABCD',['quarter','half','quarter_top','half_top']):
 p=pub['configs'][key]['aggregates'];g=final_df[final_df.arrangement==c]
 print(c,'published/corrected','180',p['t_ch5']['mean'],g.T_CFC180.mean(),'CV',p['t_ch5']['cv'],100*g.T_CFC180.std(ddof=1)/g.T_CFC180.mean(),'1000',p['t_ch5_1000']['mean'],g.T_CFC1000.mean(),'CV',p['t_ch5_1000']['cv'],100*g.T_CFC1000.std(ddof=1)/g.T_CFC1000.mean())

|Sig.|Cfg.|Raw in/out (G)|T raw|CFC1000 in/out (G)|T1000|CFC180 in/out (G)|T180|FWHM (ms)|Captured Δv (m/s)|
|---:|:--:|---:|---:|---:|---:|---:|---:|---:|---:|
|1|A|1615/1726|1.069|893.0/904.0|1.012|377.0/386.6|1.026|1.593*|5.501|
|2|A|1659/1942|1.170|843.7/927.3|1.099|369.2/380.8|1.031|1.621*|5.453|
|3|A|1793/2076|1.158|824.6/931.6|1.130|365.9/380.1|1.039|1.631*|5.422|
|4|A|1940/2154|1.110|811.1/950.9|1.172|364.2/377.7|1.037|1.639*|5.422|
|5|A|1983/2179|1.099|807.6/953.4|1.181|364.6/377.9|1.036|1.647*|5.452|
|6|A|2068/2102|1.016|795.9/957.9|1.204|362.2/375.6|1.037|1.651*|5.432|
|7|A|2230/2167|0.972|788.2/959.0|1.217|359.1/374.2|1.042|1.653*|5.377|
|8|A|2463/2185|0.887|775.6/958.3|1.236|358.9/372.5|1.038|1.664*|5.418|
|9|A|2322/2222|0.957|764.7/951.1|1.244|357.3/372.4|1.042|1.669*|5.412|
|10|A|2353/2223|0.945|759.8/950.7|1.251|356.1/372.2|1.045|1.674*|5.397|
|11|B|567/423|0.746|344.3/362.8|1.054|245.0/263.5|1.076|2.110|4.983|
|12|B|515/459|0.891|345.8/362.6|1.049|252.9/265.8|1.051|2.1

In [24]:
s=await check_task(standards_task)
print('status',s.status)
print((s.answer or '')[:30000])

Task 74990183-f4d6-4b25-8d07-dd3a48f338d6 still running (status: in progress). Call check_task(task_id) again later.
status in progress



In [25]:
from e14c.web import fetch_parsed_page
urls=['https://saemobilus.sae.org/standards/j2111_202208-instrumentation-impact-test-part-1-electronic-instrumentation','https://www.iso.org/standard/78160.html','https://www.iso.org/standard/27053.html','https://store.astm.org/d3332-99r23.html','https://store.astm.org/d7136_d7136m-25.html','https://www.iest.org/Standards-RPs/Recommended-Practices/IEST-RD-DTE012']
for u in urls:
 try:
  tx=await fetch_parsed_page(u);print('\nURL',u,'\n',tx[:2500])
 except Exception as e: print('ERR',u,type(e).__name__,str(e)[:300])


URL https://saemobilus.sae.org/standards/j2111_202208-instrumentation-impact-test-part-1-electronic-instrumentation 
 J211/1\_202208: Instrumentation for Impact Test Part 1 - Electronic Instrumentation - Recommended Practice 

[home](https://saemobilus.sae.org/ "Home")[search](https://saemobilus.sae.org/search "Search")

[layersBrowse](https://saemobilus.sae.org/browse)[library\_booksMy Library](https://saemobilus.sae.org/mylibrary)[auto\_awesomeSAE AI Chat](https://saemobilus.sae.org/ai)

[campaign](https://saemobilus.sae.org/announcements "Announcement")[help](https://saemobilus.sae.org/help "Help")

arrow\_back**Browse**

[Automotive](https://saemobilus.sae.org/search#industrysectors_name=Automotive)[Recommended Practice](https://saemobilus.sae.org/search#sub_group=Recommended%20Practice)

# J211/1\_202208 Instrumentation for Impact Test Part 1 - Electronic Instrumentation

## Revised

## 8/19/2022

[Description](https://saemobilus.sae.org/standards/j2111_202208-instrumentation-imp


URL https://www.iso.org/standard/78160.html 
 ISO 5348:2021 - Mechanical vibration and shock — Mechanical mounting of accelerometers 

* [Applications](https://login.iso.org "ISO's applications portal")
* [OBP](https://www.iso.org/obp/ui/en/ "Online Browsing Platform")
* [English](https://www.iso.org/standard/78160.html#)
    + [español](https://www.iso.org/es/contents/data/standard/07/81/78160.html)
    + [français](https://www.iso.org/fr/standard/78160.html)
    + [русский](https://www.iso.org/ru/standard/78160.html)

![ISO logo for print](https://www.iso.org/modules/isoorg-template/img/iso/iso-logo-print.gif "width=50 | height=50")

* [Standards](https://www.iso.org/standards.html "Covering almost every product, process or service imaginable, ISO makes standards used everywhere.")
* Sectors
[Health](https://www.iso.org/sectors/health "(including medicine and laboratory equipment)")
[IT & related technologies](https://www.iso.org/sectors/it-technologies "(including communication tec


URL https://www.iso.org/standard/27053.html 
 ISO 16063-21:2003 - Methods for the calibration of vibration and shock transducers — Part 21: Vibration calibration by comparison to a reference transducer 

* [Applications](https://login.iso.org "ISO's applications portal")
* [OBP](https://www.iso.org/obp/ui/en/ "Online Browsing Platform")
* [English](https://www.iso.org/standard/27053.html#)
    + [español](https://www.iso.org/es/contents/data/standard/02/70/27053.html)
    + [français](https://www.iso.org/fr/standard/27053.html)
    + [русский](https://www.iso.org/ru/standard/27053.html)

![ISO logo for print](https://www.iso.org/modules/isoorg-template/img/iso/iso-logo-print.gif "width=50 | height=50")

* [Standards](https://www.iso.org/standards.html "Covering almost every product, process or service imaginable, ISO makes standards used everywhere.")
* Sectors
[Health](https://www.iso.org/sectors/health "(including medicine and laboratory equipment)")
[IT & related technologies](http


URL https://store.astm.org/d3332-99r23.html 
 D3332 Standard Test Methods for Mechanical-Shock Fragility of Products, Using Shock Machines

The store will not work correctly when cookies are disabled.

Using Artificial Intelligence (AI) on ASTM standards and related intellectual property is prohibited. Violations will result in suspension of access.

[![](https://store.astm.org/static/version1783563591/frontend/ASTM/default/en_US/images/logo/astm.svg)![](https://store.astm.org/static/version1783563591/frontend/ASTM/default/en_US/images/logo/astm-logo-mobile.svg)](https://store.astm.org/)

[Cart](https://store.astm.org/checkout/cart/)

[Cart](https://store.astm.org/checkout/cart/)

[Sign In](https://store.astm.org/sso/login/index/) [Create Account](https://store.astm.org/customer/account/create/)

##### [ASTM Logo](https://store.astm.org/)

[Home](https://www.astm.org "Home")

###### [About ASTM Overview](https://www.astm.org/about)

[Governance](https://www.astm.org/about/governance "


URL https://store.astm.org/d7136_d7136m-25.html 
 D7136/D7136M Standard Test Method for Measuring the Damage Resistance of a Fiber-Reinforced Polymer Matrix Composite to a Drop-Weight Impact Event

The store will not work correctly when cookies are disabled.

Using Artificial Intelligence (AI) on ASTM standards and related intellectual property is prohibited. Violations will result in suspension of access.

[![](https://store.astm.org/static/version1783563591/frontend/ASTM/default/en_US/images/logo/astm.svg)![](https://store.astm.org/static/version1783563591/frontend/ASTM/default/en_US/images/logo/astm-logo-mobile.svg)](https://store.astm.org/)

[Cart](https://store.astm.org/checkout/cart/)

[Cart](https://store.astm.org/checkout/cart/)

[Sign In](https://store.astm.org/sso/login/index/) [Create Account](https://store.astm.org/customer/account/create/)

##### [ASTM Logo](https://store.astm.org/)

[Home](https://www.astm.org "Home")

###### [About ASTM Overview](https://www.astm.org/ab


URL https://www.iest.org/Standards-RPs/Recommended-Practices/IEST-RD-DTE012 
 IEST-RD-DTE012: Handbook for Dynamic Data Acquisition and Analysis




In [26]:
src = In[30]
src = src.replace("report=f'''", "report=r'''", 1)
needle = "path=Path('/workspace/edison-trajectories/pu-configs/adversarial-review.md')"
src = src.replace(needle, "report = report.replace('{perdrop_md}', perdrop_md)\n" + needle, 1)
exec(src)

/workspace/edison-trajectories/pu-configs/adversarial-review.md 32291 chars 32164 lines 282
# Adversarial review of the polyurethane absorber arrangement sweep

**Dataset:** `bpx68c`, 40 drops, 2026-07-30  
**Decision reviewed:** adopt arrangement B, 1/2 inch polyurethane alone  
**Verdict:** **Do not select B from this sweep. None of A–D is supported as the Bayesian-optimization operating point.** B remains a reasonable candidate for a randomized discrimination test, but the present data measure one specimen under four sequential input conditions. They do not measure the quantity needed for selection: separation among geometries relative to print and repeat noise.

## Conclusions that change the decision

1. **The exported records do contain usable pre-trigger data.** In every record, raw CH5 first crosses the stated trigger about 0.39 ms after `t = 0` (arrangement means 0.377–0.390 ms). At `t = 0`, raw CH5 is only 0.09–2.6% of its peak, not 22–53%. The reported 22–53% is created main

In [27]:
from pathlib import Path
p=Path('/workspace/edison-trajectories/pu-configs/adversarial-review.md')
txt=p.read_text()
# The raw-string draft preserved doubled LaTeX escapes; Markdown math needs one backslash.
txt=txt.replace('\\\\','\\')
p.write_text(txt)
print('bytes',p.stat().st_size,'lines',len(txt.splitlines()))
print('unresolved placeholder', '{perdrop_md}' in txt)
print('double-backslash count', txt.count('\\\\'))
print('per-drop rows',sum(1 for ln in txt.splitlines() if ln.startswith('|') and ln.split('|')[1].isdigit()))
print('banned-token checks',{w:txt.lower().count(w) for w in [' delve',' moreover',' pivotal',' crucial',' in summary',' overall']})
print('approximation double-tilde count',txt.count('~~'))
print('\n'.join(txt.splitlines()[100:130]))

bytes 32268 lines 281
unresolved placeholder False
double-backslash count 0
per-drop rows 40
banned-token checks {' delve': 0, ' moreover': 0, ' pivotal': 0, ' crucial': 0, ' in summary': 0, ' overall': 0}
approximation double-tilde count 0
|35|D|209/285|1.365|169.6/203.1|1.198|162.2/179.0|1.103|3.209|4.992|
|36|D|216/285|1.320|181.3/214.6|1.184|168.1/184.8|1.099|3.131|5.068|
|37|D|215/282|1.308|179.9/216.8|1.205|166.4/184.4|1.108|3.106|4.982|
|38|D|216/280|1.296|179.9/212.7|1.182|166.5/183.1|1.100|3.110|4.995|
|39|D|224/290|1.295|183.5/214.3|1.168|170.2/185.1|1.088|3.123|5.118|
|40|D|215/280|1.301|183.0/213.1|1.165|168.4/184.7|1.096|3.048|4.952|
|41|D|240/293|1.220|194.4/214.4|1.103|180.4/185.8|1.030|3.153|5.493|

Machine-readable copies accompany this report as `independent_per_drop_metrics.csv` and `independent_arrangement_summary.csv`.

## 2. Record start, filtering, and Δv

### 2.1 The record is not starting at the trigger crossing

Across all 40 files, raw CH5 crosses the configu

In [28]:
s=await check_task(standards_task)
print('status',s.status)
print((s.answer or '')[:30000])

Task 74990183-f4d6-4b25-8d07-dd3a48f338d6 still running (status: in progress). Call check_task(task_id) again later.
status in progress



In [29]:
import os
from e14c.data_storage import upload
artifact_uri=upload('/workspace/edison-trajectories/pu-configs', description='Commit-ready adversarial markdown review with independent per-drop and arrangement-level metric tables.', name='pu-configs-adversarial-review', project_id='5223cf63-fcbf-4e47-ad6c-f9d6fbb91b64')
print(artifact_uri)

Uploaded /workspace/edison-trajectories/pu-configs to data storage as entry data_entry:pu-configs-adversarial-review-96l4 under the name 'pu-configs-adversarial-review'.
data_entry:pu-configs-adversarial-review-96l4
